In [44]:
import networkx as nx
import pandas as pd
import numpy as np
from math import sqrt
from tqdm import tqdm

In [45]:
pools = pd.read_csv(
    '../data/pools/pools.csv',
    index_col=0,
    names=["index", "address", "version", "token0", "token1", "fee", "block_number", "timestamp", "tickspacing"]
).sort_index().drop_duplicates()
tokens = pd.read_csv(
    '../data/tokens/tokens.csv',
    index_col=0,
    names=["index", "address", "name", "symbol", "decimals"]
).sort_index().drop_duplicates()
print(len(pools))
print(len(tokens))

430384
406659


In [47]:
pools.head()

,address,version,token0,token1,fee,block_number,timestamp,tickspacing
index,,,,,,,,
1,0x00001bea43608c5ee487f82b773af8bd7cb20a6f,2,0x12898dabdbedd0fdc5c9ea4448d532ab55d92740,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,3000,17589578,1688097587,0
2,0x000024feb293b6c6c3a80a95f1f830a8746400b9,3,0x6aa56e1d98b3805921c170eb4b3fe7d4fda6d89b,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,10000,20425838,1722420503,200
3,0x00004ee988665cdda9a1080d5792cecd16dc1220,2,0x4d44d6c288b7f32ff676a4b2dafd625992f8ffbd,0xdac17f958d2ee523a2206206994597c13d831ec7,3000,11666388,1610801924,0
4,0x0000871c95bb027c90089f4926fd1ba82cdd9a8b,2,0x5152e1cb69a2ffa3997e89cbb4aba76a01d82141,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,3000,10986295,1601773155,0
5,0x000089906c37426585e860d02c545ab1d184a6ba,2,0x722383e1e8994cb8a55cbc1621dc068b62403b1e,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,3000,19685531,1713481823,0


In [48]:
tokens.head()

,address,name,symbol,decimals
index,,,,
0,0x12898dabdbedd0fdc5c9ea4448d532ab55d92740,Kabosu 2.0,KABOSU2.0,9
1,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,Wrapped Ether,WETH,18
2,0x6aa56e1d98b3805921c170eb4b3fe7d4fda6d89b,MAGA,TRUMP,9
3,0x4d44d6c288b7f32ff676a4b2dafd625992f8ffbd,Solution Life Coin,SLC,18
4,0xdac17f958d2ee523a2206206994597c13d831ec7,Tether USD,USDT,6


In [63]:
old_pools = pools[pools['block_number'] <= 18e6]
all_pools = old_pools[['token0', 'token1']].to_numpy().tolist()
G = nx.from_edgelist(all_pools, create_using=nx.MultiDiGraph)
print("all_pools: ", len(all_pools))
print("tokens: ", len(tokens))
print("nodes: ", len(G.nodes))
print("edges: ", len(G.edges))

remove = [node for (node, d) in dict(G.degree()).items() if d <= 15]
G.remove_nodes_from(remove)
print("after nodes: ", len(G.nodes))
print("after edges: ", len(G.edges))

all_pools:  257348
tokens:  406659
nodes:  241892
edges:  257348
after nodes:  150
after edges:  1885


In [64]:
# filter based on degree
edges = list(G.edges())
mask = []
for (t0, t1) in tqdm(all_pools):
    if (t0, t1) in edges or (t1, t0) in edges:
        mask.append(True)
    else:
        mask.append(False)
masked_pools= old_pools[mask]
#masked_pools.to_csv('../data/pools/pools_deg_filter.csv', sep=',', header=False)

100%|████████████████████████████████| 257348/257348 [00:10<00:00, 24619.01it/s]


In [65]:
filtered_tokens = list(G.nodes())
mask = []
for t in tokens['address']:
    if t in filtered_tokens:
        mask.append(True)
    else:
        mask.append(False)
masked_tokens = tokens[mask]
#masked_tokens.to_csv('../data/tokens_deg_filter_all.csv', sep=',', header=False)

In [66]:
prices = pd.read_parquet('../data/prices/prices.parquet').drop_duplicates()
prices.reset_index()
prices.head()

,pool_address,block_number,reserve_t0,reserve_t1,sqrt_price_x96
0,0x0002e63328169d7feea121f1e32e4f620abf0352,21810878,591379112586406197411,7514697609889,24980640316888636113055116
1,0x0004cd8474e882278e32e584699090be496f410e,21810878,1789432435576660297853997,21899650982,8635683066121563865695
2,0x0005ee2ef3f313ae168451ef8174b4fcc508819f,21810878,110542,10,None
3,0x000ba527862e5b82cff0f7c66b646af023274aa1,21810878,1243565337392500971972213,57124474485134695238,1166871022256290171535964128
4,0x000e0bbcfdba9490ee20b0a7900a4dff46317998,21810878,160820868314075,1213616988401802,29831852388212861267170748790


In [67]:
weth = "0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2"
weth_filter = 100e18
iter_pools = masked_pools[masked_pools['address'].isin(list(prices['pool_address']))]
mask = []
for _, pool in tqdm(iter_pools.iterrows()):
    try:
        t0_a = pool['token0']
        t1_a = pool['token1']
        pool_price = prices[prices['pool_address'] == pool.address].iloc[0]
        if t0_a == weth:
            t0_w_r = int(pool_price['reserve_t0'])
            mask.append(t0_w_r >= weth_filter)
            continue
        elif t1_a == weth:
            t1_w_r = int(pool_price['reserve_t1'])
            mask.append(t1_w_r >= weth_filter)
            continue
        else:
            append = False
            for i, token in enumerate([t0_a, t1_a]):
                try:
                    weth_pools = masked_pools[
                        ((masked_pools['token0'] == token) | (masked_pools['token1'] == token))
                        &
                        ((masked_pools['token0'] == weth) | (masked_pools['token1'] == weth))
                    ]
                    price_entries = prices[prices['pool_address'].isin(weth_pools['address'])]
                    if weth == weth_pools['token1'].iloc[0]:
                        weth_list = list(int(a) for a in price_entries['reserve_t1'])
                        max_entry = price_entries.iloc[weth_list.index(max(weth_list))]
                        reserve_in_weth = cal_reserve(
                            i,
                            pool_price, 
                            int(max_entry['reserve_t1']), 
                            int(max_entry['reserve_t0'])
                        )
                        
                    else:
                        weth_list = list(int(a) for a in price_entries['reserve_t0'])
                        max_entry = price_entries.iloc[weth_list.index(max(weth_list))]
                        reserve_in_weth = cal_reserve(
                            i,
                            pool_price, 
                            int(max_entry['reserve_t1']), 
                            int(max_entry['reserve_t0'])
                        )
                    append = reserve_in_weth >= weth_filter and reserve_in_weth <= 1e25
                except Exception as e:
                    if len(weth_pools) > 1:
                        print(f"len pools {len(weth_pools)}")
                        print(f"token {token} , {i}")
                        print(f"exception {e}, pool {pool['address']}")
                        print(f"inner excetion {e}")
                        print(f"append {append}")
                        print("-------")
                    continue
                    
        mask.append(append)
    except Exception as e:
        print(f"len pools {len(weth_pools)}")
        print(f"exception {e}, pool {pool['address']}")
        mask.append(False)
def cal_reserve(i, pool_price, up, down):
    if i == 0:
        reserve_in_weth = (int(pool_price['reserve_t0'])*up)/down
    else: 
        reserve_in_weth = (int(pool_price['reserve_t1'])*up)/down
    return reserve_in_weth
        
    

1885it [00:04, 464.07it/s]


In [68]:
final = iter_pools[mask]
print("Tokens: ", len(set(list(final['token0']) + list(final['token1']))))
print("Pools: ", len(final))
print("Non weth pools: ", len(final[(final['token0'] != weth) & (final['token1'] != weth)]))

Tokens:  64
Pools:  127
Non weth pools:  80


In [71]:
final.to_csv('../data/pools/pools_deg_15_liq_100_block_18.csv', sep=',', header=False)